# Make a synthetic cohort

This notebook generates a cohort of **synthetic patient phenopackets** with the
VAE in this repo, and writes them to `example_cohort/`. Run it top to bottom
(*Kernel → Restart & Run All*) to reproduce the cohort.

**Prereqs** (see `README.md`): `uv sync`, and a checkout of the
`exomiser-rerank-benchmark` repo for the training data. If that repo isn't at
`~/PythonProject/exomiser-rerank-benchmark`, set `SYNTHVAE_BENCH_REPO` before
launching Jupyter.

## 1. Load the real corpus
Load the Phenopacket Store and split off a holdout, exactly as the pipeline does.

In [1]:
from synthvae.data import build_corpus, stratified_split
from synthvae.eval import disease_counts

corpus = build_corpus(min_cases=20)          # diseases with >= 20 real cases
train, holdout = stratified_split(corpus, seed=0)
print(f"{len(corpus)} cases, {len(set(c.omim for c in corpus))} diseases")
print(f"{len(train)} train / {len(holdout)} holdout")

5888 cases, 99 diseases
4715 train / 1173 holdout


## 2. Train the VAE
A small conditional VAE; ~15-20s on CPU.

In [2]:
from synthvae.vae import VAEGenerator

gen = VAEGenerator(seed=0).fit(train, epochs=300)
print("VAE trained")

VAE trained


## 3. Generate the cohort
Here we make **5 cases per disease**. To instead mirror the real holdout's
per-disease counts, use `counts = disease_counts(holdout)`.

In [3]:
diseases = sorted(disease_counts(train))     # diseases the VAE learned
counts = {omim: 5 for omim in diseases}
cohort = gen.sample_cohort(counts)
print(f"generated {len(cohort)} cases across {len(counts)} diseases")
cohort[0]

generated 495 cases across 99 diseases


Case(case_id='vae_OMIM:103580_0', omim='OMIM:103580', gene='GNAS', terms=('HP:0001249', 'HP:0002905', 'HP:0004322'))

## 4. Write phenopackets to disk
We reuse the writer helpers from `scripts/generate.py` so the output is identical
to the CLI. Each `Case` becomes a Phenopacket-schema v2 JSON in `example_cohort/`.

In [4]:
import sys, json
from pathlib import Path
sys.path.insert(0, "scripts")
from generate import harvest_labels, to_phenopacket
from synthvae import config

out_dir = Path("example_cohort")
out_dir.mkdir(exist_ok=True)
hpo_labels, disease_labels = harvest_labels(config.STORE_DIR)
provenance = {"synthetic": True, "generator": "vae", "seed": 0, "epochs": 300, "min_cases": 20}

import re
for case in cohort:
    ppkt = to_phenopacket(case, hpo_labels, disease_labels, provenance)
    fname = re.sub(r"[^A-Za-z0-9._-]", "_", case.case_id) + ".json"
    (out_dir / fname).write_text(json.dumps(ppkt, indent=1))
print(f"wrote {len(cohort)} phenopackets to {out_dir}/")

wrote 495 phenopackets to example_cohort/


## 5. Peek at one generated case

In [5]:
sample = json.loads((out_dir / (re.sub(r'[^A-Za-z0-9._-]', '_', cohort[0].case_id) + '.json')).read_text())
print(json.dumps(sample, indent=1))

{
 "id": "vae_OMIM:103580_0",
 "subject": {
  "id": "vae_OMIM:103580_0"
 },
 "phenotypicFeatures": [
  {
   "type": {
    "id": "HP:0001249",
    "label": "Intellectual disability"
   }
  },
  {
   "type": {
    "id": "HP:0002905",
    "label": "Hyperphosphatemia"
   }
  },
  {
   "type": {
    "id": "HP:0004322",
    "label": "Short stature"
   }
  }
 ],
 "diseases": [
  {
   "term": {
    "id": "OMIM:103580",
    "label": "Pseudohypoparathyroidism Ia"
   }
  }
 ],
 "interpretations": [
  {
   "id": "vae_OMIM:103580_0",
   "progressStatus": "SOLVED",
   "diagnosis": {
    "disease": {
     "id": "OMIM:103580",
     "label": "Pseudohypoparathyroidism Ia"
    },
    "genomicInterpretations": [
     {
      "interpretationStatus": "CAUSATIVE",
      "variantInterpretation": {
       "variationDescriptor": {
        "geneContext": {
         "symbol": "GNAS"
        }
       }
      }
     }
    ]
   }
  }
 ],
 "metaData": {
  "createdBy": "synthvae",
  "resources": [
   {
    "id": "hp",

---
**That's the cohort.** The 495 JSON files in `example_cohort/` are the synthetic
cases. Each carries its HPO `phenotypicFeatures`, the ground-truth `diseases`
(OMIM) and causal gene, and a `_synthvae` provenance block marking it synthetic.

For other generators or sizes, see `scripts/generate.py -h` (e.g.
`--generator marginal`, `--per-disease`, `--diseases OMIM:...`).